# Week 4 Capstone: Sales Data Analysis (Python)
Cleaning, exploration and analysis of the sales dataset (700 rows, 16 columns).

## Step 1: Load the data
The `keep_default_na=False` option keeps the value 'None' in *Discount Band* (it means no discount, not a missing value). Column names are stripped because ' Sales' has a leading space.

In [2]:
import pandas as pd, glob
path = glob.glob('/kaggle/input/**/*.csv', recursive=True)[0]
print(path)
df = pd.read_csv(path, keep_default_na=False)
df.columns = df.columns.str.strip()
df.info()

/kaggle/input/datasets/prathssss/capstone/cleaned_sales.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Segment              700 non-null    object 
 1   Country              700 non-null    object 
 2   Product              700 non-null    object 
 3   Discount Band        700 non-null    object 
 4   Units Sold           700 non-null    float64
 5   Manufacturing Price  700 non-null    int64  
 6   Sale Price           700 non-null    int64  
 7   Gross Sales          700 non-null    float64
 8   Discounts            700 non-null    float64
 9   Sales                700 non-null    float64
 10  COGS                 700 non-null    float64
 11  Profit               700 non-null    float64
 12  Date                 700 non-null    object 
 13  Month Number         700 non-null    int64  
 14  Month Name           700 non-n

## Step 2: Explore the data (EDA)

In [3]:
print(df.isnull().sum().sum(), 'missing values')
print(df.duplicated().sum(), 'duplicate rows')
print(df['Discount Band'].value_counts())
print(df['Segment'].nunique(), df['Country'].nunique(), df['Product'].nunique())
print('Total Sales:', round(df['Sales'].sum(), 2))
print('Total Profit:', round(df['Profit'].sum(), 2))
print('Loss-making rows:', (df['Profit'] < 0).sum())
df.describe().round(2)

0 missing values
0 duplicate rows
Discount Band
High      245
Medium    242
Low       160
None       53
Name: count, dtype: int64
5 5 6
Total Sales: 118726350.26
Total Profit: 16893702.26
Loss-making rows: 58


,Units Sold,Manufacturing Price,Sale Price,Gross Sales,Discounts,Sales,COGS,Profit,Month Number,Year,Discount %
count,700.00,700.00,700.00,700.00,700.00,700.00,700.00,700.00,700.00,700.00,700.00
mean,1608.29,96.48,118.43,182759.43,13150.35,169609.07,145475.21,24133.86,7.90,2013.75,7.33
std,867.43,108.60,136.78,254262.28,22962.93,236726.35,203865.51,42760.63,3.38,0.43,4.60
min,200.00,3.00,7.00,1799.00,0.00,1655.08,918.00,-40617.50,1.00,2013.00,0.00
25%,905.00,5.00,12.00,17391.75,800.32,15928.00,7490.00,2805.96,5.75,2013.75,3.00
50%,1542.50,10.00,20.00,37980.00,2585.25,35540.20,22506.25,9242.20,9.00,2014.00,7.00
75%,2229.12,250.00,300.00,279025.00,15956.34,261077.50,245607.50,22662.00,10.25,2014.00,11.00
max,4492.50,260.00,350.00,1207500.00,149677.50,1159200.00,950625.00,262200.00,12.00,2014.00,15.00


## Step 3: Business questions with Pandas
Sales, profit and profit margin by Segment, Country, Product and Year.

In [4]:
for col in ['Segment', 'Country', 'Product', 'Year']:
    t = df.groupby(col)[['Sales', 'Profit']].sum().sort_values('Sales', ascending=False)
    t['Margin %'] = (t['Profit'] / t['Sales'] * 100).round(1)
    print(t.round(1))
    print()

                       Sales      Profit  Margin %
Segment                                           
Government        52504260.7  11388173.2      21.7
Small Business    42427918.5   4143168.5       9.8
Enterprise        19611694.4   -614545.6      -3.1
Midmarket          2381883.1    660103.1      27.7
Channel Partners   1800593.6   1316803.1      73.1

                               Sales     Profit  Margin %
Country                                                  
United States of America  25029830.2  2995540.7      12.0
Canada                    24887654.9  3529228.9      14.2
France                    24354172.3  3781020.8      15.5
Germany                   23505340.8  3680388.8      15.7
Mexico                    20949352.1  2907523.1      13.9

                Sales     Profit  Margin %
Product                                   
Paseo      33011144.0  4797438.0      14.5
VTT        20511921.0  3034608.0      14.8
Velo       18250059.5  2305992.5      12.6
Amarilla   17747116.

## Step 4: Why does Enterprise lose money?
All loss-making rows belong to the Enterprise segment. We check them by discount band and product, then save the cleaned data.

In [5]:
loss = df[df['Profit'] < 0]
print(loss['Segment'].value_counts())
print(loss['Discount Band'].value_counts())
print(loss['Product'].value_counts())

ent = df[df['Segment'] == 'Enterprise']
print(ent.groupby('Discount Band')[['Sales', 'Profit']].sum().round(0))

df['Discount %'] = (df['Discounts'] / df['Gross Sales'] * 100).round(1)
print(df.groupby('Segment')['Discount %'].mean().round(1))

df.to_csv('cleaned_sales.csv', index=False)
print(df.shape)

Segment
Enterprise    58
Name: count, dtype: int64
Discount Band
High      33
Medium    25
Name: count, dtype: int64
Product
Velo         13
Carretera    12
Paseo        11
Amarilla      9
VTT           8
Montana       5
Name: count, dtype: int64
                   Sales    Profit
Discount Band                     
High           6528971.0 -622369.0
Low            6442216.0  101776.0
Medium         5115508.0 -154952.0
None           1525000.0   61000.0
Segment
Channel Partners    7.1
Enterprise          6.6
Government          7.5
Midmarket           7.6
Small Business      7.5
Name: Discount %, dtype: float64
(700, 17)


## Key findings
- Total sales 118.7M, profit 16.9M (about 14% margin).
- Government and Small Business bring in most of the sales.
- Enterprise loses money (all 58 loss-making rows), mainly on High and Medium discount orders.
- Channel Partners have a very high margin but small sales.
- 2013 has only 4 months of data (Sep to Dec), so year comparisons need care.